# Middleware (LangChain 1.3+)

**Middleware** hooks into the agent loop so you can control what happens **inside** `create_agent` — without rewriting the graph yourself.

**Use middleware for:**

- Logging, analytics, debugging
- Transforming prompts, tool selection, output formatting
- Retries, fallbacks, early termination
- Rate limits, guardrails, PII detection
- **Human approval** before risky tool calls

**Prerequisite:** `OPENAI_API_KEY` in repo-root `.env`. Prior: [1_lang_chain_intro.ipynb](1_lang_chain_intro.ipynb).

**Sections:** 1 design | 2 hooks | 3 built-in | 4 summarization (messages) | 5 custom hooks | 6–7 summarization (tokens, fraction) | 8 human-in-the-loop | 9 summary

## 1. Architecture design

Training reference: **[reference/middleware-design.png](reference/middleware-design.png)**

### Agent (no middleware)

**request → model ⇄ tools → result**

### Agent with middleware (hooks)

```mermaid
%%{init: {"theme":"base","themeVariables":{"primaryColor":"#ffffff","primaryTextColor":"#111827","primaryBorderColor":"#374151","lineColor":"#374151","mainBkg":"#ffffff","nodeTextColor":"#111827"}}}%%
flowchart TD
  REQ[request] --> BA[before_agent]
  BA --> BM[before_model]
  BM --> WM[wrap_model_call]
  WM --> MODEL[model]
  MODEL --> AM[after_model]
  AM -->|tool_calls| WT[wrap_tool_call]
  WT --> TOOLS[tools]
  TOOLS --> BM
  AM -->|final answer| AA[after_agent]
  AA --> RES[result]
  style BA fill:#fff7ed,stroke:#ea580c,color:#111827
  style BM fill:#fff7ed,stroke:#ea580c,color:#111827
  style WM fill:#fff7ed,stroke:#ea580c,color:#111827
  style WT fill:#fff7ed,stroke:#ea580c,color:#111827
  style AM fill:#fff7ed,stroke:#ea580c,color:#111827
  style AA fill:#fff7ed,stroke:#ea580c,color:#111827
  style MODEL fill:#fffbeb,stroke:#d97706,color:#111827
  style TOOLS fill:#fdf2f8,stroke:#db2777,color:#111827
  style RES fill:#ecfdf5,stroke:#059669,color:#111827
```

Pass middleware to `create_agent(..., middleware=[...])`. Multiple middleware runs in list order.

## 2. Hook reference

| Hook | Runs when | Typical uses |
|------|-----------|--------------|
| **`before_agent`** | Once, before the graph starts | Session setup, input validation |
| **`before_model`** | Before each LLM call | Trim/summarize messages, rate limits |
| **`wrap_model_call`** | Around the LLM invoke | Retries, fallbacks, caching |
| **`after_model`** | After each LLM response | Logging, guardrails |
| **`wrap_tool_call`** | Around each tool execution | Retries, sandboxing, **human approval** |
| **`after_agent`** | Once, after the graph finishes | Analytics, cleanup |

Subclass `AgentMiddleware` or use decorators (`@before_model`, etc.) from `langchain.agents.middleware`.

## 3. Built-in middleware

| Class | Hook | Purpose |
|-------|------|---------|
| `SummarizationMiddleware` | `before_model` | Compress old messages when history grows |
| `HumanInTheLoopMiddleware` | `wrap_tool_call` | Pause for approve / edit / reject |
| `PIIMiddleware` | `before_model` | Detect / redact PII |
| `ModelRetryMiddleware` | `wrap_model_call` | Retry failed model calls |
| `ModelFallbackMiddleware` | `wrap_model_call` | Backup models on error |
| `ModelCallLimitMiddleware` | `before_model` | Cap LLM calls per run |
| `ToolRetryMiddleware` | `wrap_tool_call` | Retry failed tools |

In [ ]:
"""Load API keys from .env."""
import os
from pathlib import Path

from dotenv import load_dotenv

for env_path in (Path.cwd() / ".env", Path.cwd().parent / ".env"):
    if env_path.is_file():
        load_dotenv(env_path)
        break
else:
    load_dotenv()

value = os.getenv("OPENAI_API_KEY")
if value:
    os.environ["OPENAI_API_KEY"] = value
else:
    print("OPENAI_API_KEY: missing (add to .env)")

## 4. `SummarizationMiddleware` — trigger by **message count**

Requires **`checkpointer`** + stable **`thread_id`** so history accumulates across `invoke` calls.

| Param | Example | Meaning |
|-------|---------|---------|
| `trigger` | `("messages", 10)` | Summarize when ≥ 10 messages |
| `keep` | `("messages", 4)` | Keep last 4 messages verbatim |
| `model` | `"gpt-4o-mini"` | Model that writes the summary |

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver

agent = create_agent(
    model="gpt-4o-mini",
    tools=[],
    system_prompt="You are a helpful assistant.",
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="gpt-4o-mini",
            trigger=("messages", 10),
            keep=("messages", 4),
        )
    ],
)
print("Summarization agent created.")

In [ ]:
config = {"configurable": {"thread_id": "middleware-demo-1"}}

result = agent.invoke(
    {"messages": [{"role": "user", "content": "Remember: my favorite color is blue."}]},
    config=config,
)
result["messages"][-1].content

### 4b. Multi-turn demo — watch summarization fire

Six math questions on one thread. Count goes 2 → 4 → … → **10**, then drops to **6** when older turns become one summary `HumanMessage` (`additional_kwargs: lc_source=summarization`).

In [ ]:
from langchain_core.messages import HumanMessage

config = {"configurable": {"thread_id": "test-messages-1"}}

for q in [
    "What is 2+2?",
    "What is 10*5?",
    "What is 100/4?",
    "What is 15-7?",
    "What is 3*3?",
    "What is 4*4?",
]:
    response = agent.invoke({"messages": [HumanMessage(content=q)]}, config)
    print(f"Q: {q} | messages: {len(response['messages'])} | reply: {response['messages'][-1].content[:50]}...")

## 5. Custom logging middleware

Decorators return `AgentMiddleware`. Return `None` to leave state unchanged.

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import after_agent, before_model


@before_model
def log_before_model(state, runtime):
    print(f"[before_model] {len(state['messages'])} messages")
    return None


@after_agent
def log_after_agent(state, runtime):
    print(f"[after_agent] done, {len(state['messages'])} messages")
    return None


logging_agent = create_agent(
    model="gpt-4o-mini",
    tools=[],
    middleware=[log_before_model, log_after_agent],
)
logging_agent.invoke({"messages": [{"role": "user", "content": "Say hi in one word."}]})

## 6. Summarization — trigger by **token count** (+ tools)

Use `("tokens", N)` when tool outputs are long. Each hotel search adds ~4 messages (Human → AI tool_calls → ToolMessage → AI answer).

| Param | Example | Meaning |
|-------|---------|---------|
| `trigger` | `("tokens", 550)` | Summarize when history ≥ ~550 tokens |
| `keep` | `("tokens", 200)` | Keep ~200 tokens of recent context |

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain_core.messages import HumanMessage
from langchain_core.tools import tool
from langgraph.checkpoint.memory import InMemorySaver


@tool
def search_hotels(city: str) -> str:
    """Search hotels — returns a long response to consume tokens."""
    return f"""Hotels in {city}:
    1. Grand Hotel - 5 star, $350/night, spa, pool, gym
    2. City Inn - 4 star, $180/night, business center
    3. Budget Stay - 3 star, $75/night, free wifi"""


hotel_agent = create_agent(
    model="gpt-4o-mini",
    tools=[search_hotels],
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="gpt-4o-mini",
            trigger=("tokens", 550),
            keep=("tokens", 200),
        ),
    ],
)
config_tokens = {"configurable": {"thread_id": "hotel-demo-1"}}


def count_tokens(messages):
    return sum(len(str(m.content)) for m in messages) // 4

In [ ]:
for city in ["Paris", "London", "Tokyo", "New York", "Dubai", "Singapore"]:
    response = hotel_agent.invoke(
        {"messages": [HumanMessage(content=f"Find hotels in {city}")]},
        config=config_tokens,
    )
    summarized = any(
        getattr(m, "additional_kwargs", {}).get("lc_source") == "summarization"
        for m in response["messages"]
    )
    print(
        f"{city}: ~{count_tokens(response['messages'])} tokens, "
        f"{len(response['messages'])} msgs, summarized={summarized}"
    )

## 7. Summarization — trigger by **fraction** (context window %)

Uses a **percentage of the model's context window** instead of fixed counts. Good when you want summarization relative to model limits.

| Param | Example | Meaning |
|-------|---------|---------|
| `trigger` | `("fraction", 0.005)` | Summarize at 0.5% of context (~640 tokens for 128k) |
| `keep` | `("fraction", 0.002)` | Keep 0.2% of context (~256 tokens) |

**Note:** Low fractions are useful for demos so summarization fires quickly.

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain_core.messages import HumanMessage
from langchain_core.tools import tool
from langgraph.checkpoint.memory import InMemorySaver


@tool
def search_hotels_short(city: str) -> str:
    """Search hotels."""
    return f"Hotels in {city}: Grand Hotel $350, City Inn $180, Budget Stay $75"


fraction_agent = create_agent(
    model="gpt-4o-mini",
    tools=[search_hotels_short],
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="gpt-4o-mini",
            trigger=("fraction", 0.005),
            keep=("fraction", 0.002),
        ),
    ],
)
config_fraction = {"configurable": {"thread_id": "fraction-demo-1"}}
CONTEXT = 128_000  # gpt-4o-mini approximate context

for city in ["Paris", "London", "Tokyo", "New York", "Dubai", "Singapore"]:
    response = fraction_agent.invoke(
        {"messages": [HumanMessage(content=f"Hotels in {city}")]},
        config=config_fraction,
    )
    tokens = count_tokens(response["messages"])
    pct = tokens / CONTEXT
    summarized = any(
        getattr(m, "additional_kwargs", {}).get("lc_source") == "summarization"
        for m in response["messages"]
    )
    print(f"{city}: ~{tokens} tokens ({pct:.4%}), {len(response['messages'])} msgs, summarized={summarized}")

## 8. `HumanInTheLoopMiddleware` — approve, reject, edit

Pauses the agent **before** configured tools run. The human can:

| Decision | Effect |
|----------|--------|
| **`approve`** | Run tool with LLM-proposed args |
| **`reject`** | Skip tool; agent gets error `ToolMessage` |
| **`edit`** | Run tool with **human-corrected** args |

**Good for:** database writes, payments, emails, compliance workflows.

**Flow:**
1. `agent.invoke(...)` → may return `__interrupt__` in result
2. Human reviews `action_requests`
3. `agent.invoke(Command(resume={...}), config)` → continues execution

**Requires:** `checkpointer` + same `thread_id` for resume.

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langchain_core.messages import HumanMessage
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command


def read_email_tool(email_id: str) -> str:
    """Read an email by ID."""
    return f"Email content for ID: {email_id}"


def send_email_tool(recipient: str, subject: str, body: str) -> str:
    """Send an email."""
    return f"Email sent to {recipient} with subject '{subject}'"


hitl_agent = create_agent(
    model="gpt-4o",
    tools=[read_email_tool, send_email_tool],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email_tool": {
                    "allowed_decisions": ["approve", "edit", "reject"],
                },
                "read_email_tool": False,  # auto-run, no interrupt
            }
        ),
    ],
)
print("Human-in-the-loop agent created.")

### 8a. Approve

Agent proposes `send_email_tool` → graph pauses → human approves → email sends.

In [ ]:
config_approve = {"configurable": {"thread_id": "test-approve"}}

result = hitl_agent.invoke(
    {
        "messages": [
            HumanMessage(
                content="Send email to john@test.com with subject 'Hello' and body 'How are you?'"
            )
        ]
    },
    config=config_approve,
)

if "__interrupt__" in result:
    print("Paused for approval:", result["__interrupt__"])
    result = hitl_agent.invoke(
        Command(resume={"decisions": [{"type": "approve"}]}),
        config=config_approve,
    )

result["messages"][-1].content

### 8b. Reject

Human rejects the tool call — no email is sent; agent explains cancellation.

In [ ]:
config_reject = {"configurable": {"thread_id": "test-reject"}}

result = hitl_agent.invoke(
    {
        "messages": [
            HumanMessage(
                content="Send email to john@test.com with subject 'Hello' and body 'How are you?'"
            )
        ]
    },
    config=config_reject,
)

if "__interrupt__" in result:
    print("Paused — rejecting tool call...")
    result = hitl_agent.invoke(
        Command(resume={"decisions": [{"type": "reject"}]}),
        config=config_reject,
    )

result["messages"][-1].content

### 8c. Edit then approve

Human fixes wrong recipient/subject/body before the tool runs.

In [ ]:
config_edit = {"configurable": {"thread_id": "test-edit"}}

result = hitl_agent.invoke(
    {
        "messages": [
            HumanMessage(
                content="Send email to wrong@email.com with subject 'Test' and body 'Hello'"
            )
        ]
    },
    config=config_edit,
)

if "__interrupt__" in result:
    print("Paused — editing args before send...")
    result = hitl_agent.invoke(
        Command(
            resume={
                "decisions": [
                    {
                        "type": "edit",
                        "edited_action": {
                            "name": "send_email_tool",
                            "args": {
                                "recipient": "correct@email.com",
                                "subject": "Corrected Subject",
                                "body": "This was edited by human before sending",
                            },
                        },
                    }
                ]
            }
        ),
        config=config_edit,
    )

result["messages"][-1].content

## 9. Notebook summary

### Middleware map

| Middleware | Hook | This notebook |
|------------|------|----------------|
| `SummarizationMiddleware` | `before_model` | §4 messages, §6 tokens, §7 fraction |
| Custom `@before_model` / `@after_agent` | hooks | §5 logging |
| `HumanInTheLoopMiddleware` | `wrap_tool_call` | §8 approve / reject / edit |

### Summarization triggers

| Style | Example | Best when |
|-------|---------|-----------|
| **Messages** | `("messages", 10)` | Chat turns, even-sized history |
| **Tokens** | `("tokens", 550)` | Long tool outputs |
| **Fraction** | `("fraction", 0.005)` | Relative to model context window |

### Requirements checklist

```python
from langchain.agents import create_agent
from langchain.agents.middleware import (
    SummarizationMiddleware,
    HumanInTheLoopMiddleware,
    before_model,
)
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command

# Summarization + memory
agent = create_agent(
    model="gpt-4o-mini",
    tools=[...],
    checkpointer=InMemorySaver(),
    middleware=[SummarizationMiddleware(model="gpt-4o-mini", trigger=..., keep=...)],
)
config = {"configurable": {"thread_id": "unique-id"}}
agent.invoke({"messages": [...]}, config=config)

# Human-in-the-loop
hitl = create_agent(
    model="gpt-4o",
    tools=[...],
    checkpointer=InMemorySaver(),
    middleware=[HumanInTheLoopMiddleware(interrupt_on={...})],
)
result = hitl.invoke({...}, config=config)
if "__interrupt__" in result:
    result = hitl.invoke(Command(resume={"decisions": [{"type": "approve"}]}), config=config)
```

### Tutorial series

| Notebook | Topic |
|----------|-------|
| [1_lang_chain_intro](1_lang_chain_intro.ipynb) | `create_agent`, ReAct loop |
| [3_tools](3_tools.ipynb) | Tools + manual loop |
| [4_messages](4_messages.ipynb) | Message types |
| **6_middleware** (this) | Hooks, summarization, human approval |

**Design diagram:** [reference/middleware-design.png](reference/middleware-design.png)